This is the file that you will submit. All code in here and only here will be graded.

Any analysis/write-up can also be done in here in clearly labeled markdown cells.

In [ ]:
## Setup
import os
import random
import re
import pandas as pd
import numpy as np

In [ ]:
DATA_FOLDER = 'Data/'

rooms_df = pd.read_csv(DATA_FOLDER + 'final_room_data.csv', index_col=False)

student_dfs = dict()
for i in range(1, 8):
    student_dfs[i] = pd.read_csv(DATA_FOLDER + 'student_dataset' + str(i) + '.csv', index_col=False)

print('room rows:', len(rooms_df))
print('students per dataset:', [len(student_dfs[i]) for i in range(1, 8)])

room rows: 5330
students per dataset: [21568, 22703, 22008, 24803, 20541, 21145, 21568]


In [3]:
GRAD_YEARS = ['Masters', 'Doctorate', 'Postdoctoral studies']
GRAD_HALLS = ['BA', 'JA']

In [4]:
def parse_halls(cell):
    return re.findall(r"'([^']+)'", str(cell))

def parse_roommates(cell):
    return [int(x) for x in re.findall(r"\d+", str(cell))]

In [5]:
rooms = rooms_df.copy()
rooms['building_key'] = rooms['building_id'].fillna('').astype(str).str.strip()
rooms['room_id'] = rooms['room_id'].astype(str).str.strip()
rooms['capacity'] = rooms['max_room_occupancy']
rooms['is_single'] = rooms['max_room_occupancy'] == 1
rooms['is_grad_hall'] = rooms['hall_id'].isin(GRAD_HALLS)

print('duplicate rooms:', rooms.duplicated(subset=['hall_id', 'building_key', 'room_id']).sum())
print('total beds:', rooms['capacity'].sum())

duplicate rooms: 0
total beds: 21568


In [6]:
hall_summary = rooms.groupby(['hall_id', 'hall_name']).agg(
    residence_type=('residence_type', 'first'),
    is_grad_hall=('is_grad_hall', 'first'),
    beds=('capacity', 'sum'),
    accessible=('has_accessibility_ramps', 'first'),
    single_rooms=('is_single', 'sum'),
    price=('price', 'first'),
)
hall_summary

,,residence_type,is_grad_hall,beds,accessible,single_rooms,price
hall_id,hall_name,,,,,,
AH,Allen Hall,Traditional Residence Hall,False,267,0,67,9241
BA,Buell Apartments,Apartment,True,1536,0,0,11618
BH,Barr Hall,Traditional Residence Hall,False,267,0,67,4135
BS,BEST Suites,Suite,False,3904,1,0,7523
CS,Crosby Suites,Suite,False,1080,0,0,9490
JA,Johnson Apartments,Apartment,True,400,0,0,9960
JS,Judson Suites,Suite,False,1080,0,0,10072
MH,Mattia Hall,Traditional Residence Hall,False,435,0,11,8110
MRS,Morrow Suites,Suite,False,1080,0,0,8387


In [7]:
def clean_students(df):
    students = df.copy()
    students = students.drop_duplicates(subset='RUID')
    students['is_grad'] = students['student_year'].isin(GRAD_YEARS)
    students['hall_prefs'] = students['preferred_hall_ids'].map(parse_halls)
    students['roommates'] = students['roommate_preferences'].map(parse_roommates)
    return students

students_clean = dict()
for i in range(1, 8):
    students_clean[i] = clean_students(student_dfs[i])
    print('dataset', i, '| students:', len(students_clean[i]),
          '| duplicates removed:', len(student_dfs[i]) - len(students_clean[i]),
          '| missing values:', student_dfs[i].isna().sum().sum(),
          '| grad:', students_clean[i]['is_grad'].sum(),
          '| need accessible:', students_clean[i]['accessibility_need'].sum(),
          '| want single:', students_clean[i]['is_single_preferred'].sum(),
          '| listed roommates:', (students_clean[i]['roommates'].map(len) > 0).sum())

dataset 1 | students: 21568 | duplicates removed: 0 | missing values: 0 | grad: 1936 | need accessible: 2624 | want single: 5495 | listed roommates: 16170


dataset 2 | students: 22703 | duplicates removed: 0 | missing values: 0 | grad: 2038 | need accessible: 1146 | want single: 5666 | listed roommates: 17022


dataset 3 | students: 22008 | duplicates removed: 0 | missing values: 0 | grad: 2400 | need accessible: 1093 | want single: 5527 | listed roommates: 16503


dataset 4 | students: 24803 | duplicates removed: 0 | missing values: 0 | grad: 2226 | need accessible: 1225 | want single: 6148 | listed roommates: 18600


dataset 5 | students: 20541 | duplicates removed: 0 | missing values: 0 | grad: 1844 | need accessible: 1057 | want single: 5065 | listed roommates: 15402


dataset 6 | students: 21145 | duplicates removed: 0 | missing values: 0 | grad: 1898 | need accessible: 1069 | want single: 5277 | listed roommates: 17931


dataset 7 | students: 21568 | duplicates removed: 0 | missing values: 0 | grad: 1936 | need accessible: 1082 | want single: 5429 | listed roommates: 16173


In [8]:
print('room columns:')
print(list(rooms.columns))
print()
print('student columns:')
print(list(students_clean[1].columns))

room columns:
['hall_id', 'hall_name', 'residence_type', 'building_id', 'number_of_floors', 'room_number', 'room_id', 'floor_number', 'floor_plan', 'room_type', 'max_room_occupancy', 'distance_from_student_center', 'average_room_size', 'room_contract_type', 'is_aircooled', 'has_vending_machine', 'has_laundry', 'has_main_lounge', 'has_floor_lounge', 'has_tv_in_building', 'has_recreational_activities', 'has_wardrobe', 'has_accessibility_ramps', 'has_private_bathroom', 'price', 'building_key', 'capacity', 'is_single', 'is_grad_hall']

student columns:
['name', 'sex', 'RUID', 'date_of_birth', 'student_year', 'student_school', 'accessibility_need', 'low_income_status', 'preferred_hall_ids', 'preferred_residence_type', 'is_single_preferred', 'is_private_bathroom_preferred', 'laundry_availibility', 'preferred_price_range', 'max_price', 'roommate_preferences', 'is_grad', 'hall_prefs', 'roommates']


## Room kinds

In [9]:
rooms['kind'] = rooms['hall_id'] + ' | ' + rooms['floor_plan'] + ' | ' + rooms['capacity'].astype(str)

kinds_df = rooms.groupby('kind').agg(
    hall_id=('hall_id', 'first'),
    residence_type=('residence_type', 'first'),
    floor_plan=('floor_plan', 'first'),
    capacity=('capacity', 'first'),
    num_rooms=('room_id', 'size'),
    price=('price', 'first'),
    accessible=('has_accessibility_ramps', 'first'),
    is_grad_hall=('is_grad_hall', 'first'),
)
kinds_df['single_bedroom'] = kinds_df['floor_plan'] == 'Traditional Single'
kinds_df['beds'] = kinds_df['capacity'] * kinds_df['num_rooms']

kinds = kinds_df.to_dict('index')
kinds_df

,hall_id,residence_type,floor_plan,capacity,num_rooms,price,accessible,is_grad_hall,single_bedroom,beds
kind,,,,,,,,,,
AH | Traditional Double | 2,AH,Traditional Residence Hall,Traditional Double,2,100,9241,0,False,False,200
AH | Traditional Single | 1,AH,Traditional Residence Hall,Traditional Single,1,67,9241,0,False,True,67
BA | Traditional Single | 4,BA,Apartment,Traditional Single,4,384,11618,0,True,True,1536
BH | Traditional Double | 2,BH,Traditional Residence Hall,Traditional Double,2,100,4135,0,False,False,200
BH | Traditional Single | 1,BH,Traditional Residence Hall,Traditional Single,1,67,4135,0,False,True,67
BS | Traditional Double | 4,BS,Suite,Traditional Double,4,976,7523,1,False,False,3904
CS | Traditional Double | 6,CS,Suite,Traditional Double,6,180,9490,0,False,False,1080
JA | Traditional Single | 2,JA,Apartment,Traditional Single,2,200,9960,0,True,True,400
JS | Traditional Double | 6,JS,Suite,Traditional Double,6,180,10072,0,False,False,1080


## Constraints

In [10]:
def can_live_in(student, kind):
    if student['is_grad'] != kind['is_grad_hall']:
        return False
    if student['accessibility_need'] == 1 and kind['accessible'] == 0:
        return False
    if kind['price'] > student['max_price']:
        return False
    return True

## Preferences

In [11]:
def preference_score(student, kind):
    if kind['hall_id'] in student['hall_prefs']:
        hall_score = student['hall_prefs'].index(kind['hall_id'])
    else:
        hall_score = len(student['hall_prefs'])

    if kind['residence_type'] == student['preferred_residence_type']:
        type_score = 0
    else:
        type_score = 1

    if kind['single_bedroom'] == bool(student['is_single_preferred']):
        single_score = 0
    else:
        single_score = 1

    return (hall_score, type_score, single_score, kind['price'])


def rank_kinds(student, options):
    return sorted(options, key=lambda name: preference_score(student, kinds[name]))

## Roommate groups

In [12]:
def find_groups(people):
    groups = []
    seen = set()
    for ruid in people:
        if ruid in seen:
            continue
        members = [ruid]
        seen.add(ruid)
        for person in members:
            for roommate in people[person]['roommates']:
                if roommate in people and roommate not in seen and person in people[roommate]['roommates']:
                    members.append(roommate)
                    seen.add(roommate)
        groups.append(members)
    return groups


def group_choices(members, people):
    options = []
    for name, kind in kinds.items():
        if len(members) > kind['capacity']:
            continue
        if all(can_live_in(people[ruid], kind) for ruid in members):
            options.append(name)

    total_rank = {name: 0 for name in options}
    for ruid in members:
        for position, name in enumerate(rank_kinds(people[ruid], options)):
            total_rank[name] += position

    return sorted(options, key=lambda name: total_rank[name])


def make_unit(members, people, rng):
    return {
        'members': members,
        'choices': group_choices(members, people),
        'next': 0,
        'lottery': rng.random(),
        'access': any(people[ruid]['accessibility_need'] == 1 for ruid in members),
        'budget': min(people[ruid]['max_price'] for ruid in members),
    }

## Priorities

In [13]:
def priority_budget(unit):
    access_rank = 0 if unit['access'] else 1
    return (access_rank, unit['budget'], unit['lottery'])


def priority_lottery(unit):
    access_rank = 0 if unit['access'] else 1
    return (access_rank, unit['lottery'])

## Deferred acceptance

In [14]:
def take_room(open_rooms, size):
    spaces = [space for space, count in open_rooms.items() if space >= size and count > 0]
    if not spaces:
        return False
    space = min(spaces)
    open_rooms[space] -= 1
    open_rooms[space - size] = open_rooms.get(space - size, 0) + 1
    return True


def deferred_acceptance(units, people, priority, rng):
    held = {name: [] for name in kinds}
    waiting = list(units)

    while waiting:
        applicants = {name: [] for name in kinds}
        next_waiting = []

        for unit in waiting:
            if unit['next'] < len(unit['choices']):
                name = unit['choices'][unit['next']]
                unit['next'] += 1
                applicants[name].append(unit)
            elif len(unit['members']) > 1:
                for ruid in unit['members']:
                    next_waiting.append(make_unit([ruid], people, rng))

        for name, kind in kinds.items():
            if not applicants[name]:
                continue
            everyone = held[name] + applicants[name]
            everyone.sort(key=priority)

            beds_left = kind['beds']
            open_rooms = {kind['capacity']: kind['num_rooms']}
            held[name] = []
            for unit in everyone:
                size = len(unit['members'])
                if size > beds_left:
                    next_waiting.append(unit)
                elif size > 1 and not take_room(open_rooms, size):
                    next_waiting.append(unit)
                else:
                    held[name].append(unit)
                    beds_left -= size

        waiting = next_waiting

    return held

## Handing out actual rooms

In [15]:
def place_in_rooms(held, people):
    assignment = {}
    for name, units in held.items():
        kind_rooms = rooms[rooms['kind'] == name]
        labels = [[row['hall_id'], row['building_id'], row['room_id']] for _, row in kind_rooms.iterrows()]
        free = list(kind_rooms['capacity'])
        room_sex = [None] * len(free)

        for unit in units:
            size = len(unit['members'])
            if size == 1:
                continue
            fits = [i for i in range(len(free)) if free[i] >= size]
            i = min(fits, key=lambda i: free[i])
            for ruid in unit['members']:
                assignment[ruid] = labels[i]
            free[i] -= size
            room_sex[i] = people[unit['members'][0]]['sex']

        line = {}
        for unit in units:
            if len(unit['members']) == 1:
                ruid = unit['members'][0]
                line.setdefault(people[ruid]['sex'], []).append(ruid)

        for i in range(len(free)):
            while free[i] > 0 and any(line.values()):
                sex = room_sex[i]
                if not line.get(sex):
                    sex = max(line, key=lambda s: len(line[s]))
                ruid = line[sex].pop()
                assignment[ruid] = labels[i]
                free[i] -= 1
                room_sex[i] = sex

    return assignment

## Running the datasets 

In [ ]:
def run_match(students, priority, seed):
    rng = random.Random(seed)
    people = students.set_index('RUID').to_dict('index')
    units = [make_unit(members, people, rng) for members in find_groups(people)]
    held = deferred_acceptance(units, people, priority, rng)
    assignment = place_in_rooms(held, people)
    return pd.DataFrame({
        'RUID': students['RUID'],
        'Assigned Rooms': [assignment.get(ruid, np.nan) for ruid in students['RUID']],
    })


POLICIES = {'budget': priority_budget, 'lottery': priority_lottery}
SEEDS = [1, 2, 3]

results = {}
for policy_name, priority in POLICIES.items():
    os.makedirs('results/policy_' + policy_name, exist_ok=True)
    for i in range(1, 8):
        for seed in SEEDS:
            match = run_match(students_clean[i], priority, seed)
            results[(policy_name, i, seed)] = match
            match.to_csv(f'results/policy_{policy_name}/allocated_dataset{i}_run{seed}.csv', na_rep='nan')
            assigned = match['Assigned Rooms'].notna().mean()
            print(f'policy {policy_name} | dataset {i} | run {seed} | assigned {assigned:.1%}')

policy budget | dataset 1 | run 1 | assigned 85.9%


policy budget | dataset 1 | run 2 | assigned 85.8%


policy budget | dataset 1 | run 3 | assigned 85.8%


policy budget | dataset 2 | run 1 | assigned 62.7%


policy budget | dataset 2 | run 2 | assigned 62.8%


policy budget | dataset 2 | run 3 | assigned 62.8%


... (32 outputs left out)

policy lottery | dataset 6 | run 3 | assigned 78.2%


policy lottery | dataset 7 | run 1 | assigned 79.0%


policy lottery | dataset 7 | run 2 | assigned 78.9%


policy lottery | dataset 7 | run 3 | assigned 79.0%


## Sanity check

In [ ]:
# just making sure the algorithm works properly
def check_match(students, match):
    people = students.set_index('RUID').to_dict('index')
    room_lookup = rooms.set_index(['hall_id', 'room_id', 'building_key']).to_dict('index')
    problems = 0

    placed = match.dropna(subset=['Assigned Rooms'])
    room_keys = placed['Assigned Rooms'].map(lambda r: (r[0], r[2], '' if pd.isna(r[1]) else str(r[1]).strip()))
    for key, count in room_keys.value_counts().items():
        if count > room_lookup[key]['capacity']:
            problems += 1

    for ruid, key in zip(placed['RUID'], room_keys):
        room = room_lookup[key]
        student = people[ruid]
        if student['is_grad'] != room['is_grad_hall']:
            problems += 1
        if student['accessibility_need'] == 1 and room['has_accessibility_ramps'] == 0:
            problems += 1
        if room['price'] > student['max_price']:
            problems += 1

    return problems


for (policy_name, i, seed), match in results.items():
    print(policy_name, i, seed, 'problems:', check_match(students_clean[i], match))

budget 1 1 problems: 0


budget 1 2 problems: 0


budget 1 3 problems: 0


budget 2 1 problems: 0


budget 2 2 problems: 0


budget 2 3 problems: 0


... (32 outputs left out)

lottery 6 3 problems: 0


lottery 7 1 problems: 0


lottery 7 2 problems: 0


lottery 7 3 problems: 0
